# LLaVA (1.5-7B) Image Phrase Extractor — Google Colab (Google Drive)


**Model:** llava-hf/llava-1.5-7b-hf — publicly available (no special access required)

Link:https://huggingface.co/llava-hf/llava-1.5-7b-hf


**Output per image (saved to phrases_paligemma.csv):**

**filename	phrase	meaningful_tokens	word_count	meaningful_token_count**

---


*   **word_count** — total words in the phrase
*   **meaningful_token_count** — words after removing stopwords (a, the, is, in, of, ...)


### **Before running**:
* Go to Google Drive and upload your image folder
* Set image_folder in Cell 5 to the correct path
* Allow popups for Colab (needed for Drive mount)

**If Drive mount fails:**
* Enable popups and re-run Cell 1
* Or: Runtime → Restart session → run again

**Runtime recommendation:** Runtime → Change runtime type → T4 GPU
* LLaVA 1.5-7B is a large model (~7B parameters)
* If memory issues occur → enable load_in_4bit=True in Cell 3

In [ ]:
# ── Cell 1: Mount Google Drive and set image folder ───────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

# ↓ Change this to the name of the folder you uploaded to My Drive
DRIVE_IMAGE_FOLDER = 'fire_detection/PickTheModelDataSet(30pics)'

IMAGE_DIR = f'/content/drive/MyDrive/{DRIVE_IMAGE_FOLDER}'

assert os.path.isdir(IMAGE_DIR), (
    f'Folder not found: {IMAGE_DIR}\n'
    f'Make sure it exists in My Drive.'
)

image_files = [
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".jpg", ".png", ".jpeg", "webp")) and not f.startswith("._")
]

print(f'Found {len(image_files)} images')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
!pip install -q transformers accelerate bitsandbytes spacy
!python -m spacy download en_core_web_sm

# ⚠️ After this cell finishes, go to Runtime → Restart session,
# then run all cells again from Cell 1.
print('Done. Now go to Runtime → Restart session, then re-run all cells.')

In [ ]:
# ── Cell 3: Load LLaVA (1.5-7B) model ────────────────────────────────────────

import torch
from transformers import LlavaForConditionalGeneration, AutoProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_ID = "llava-hf/llava-1.5-7b-hf"

print(f"Loading {MODEL_ID} on {device}...")

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
    # load_in_4bit=True  # включи если не хватает памяти
)

print("Model ready")

In [ ]:
# ── Cell 4: Load spaCy and helper functions ────────────────────────────────────────
import spacy
nlp = spacy.load("en_core_web_sm")

def extract_meaningful_tokens(text):
    doc = nlp(text)
    return [t.text for t in doc if t.pos_ in ["NOUN", "VERB", "ADJ"]]

In [ ]:
# ── Cell 5: Inference function ─────────────────────────────────────────────────────
from PIL import Image

prompt = (
    "Describe this image in one short phrase focusing on "
    "whether fire is present, its type, and environment."
)

def analyze_image(image_path, prompt):
    image = Image.open(image_path).convert("RGB")

    # LLaVA chat format
    formatted_prompt = f"USER: <image>\n{prompt}\nASSISTANT:"

    inputs = processor(
        text=formatted_prompt,
        images=image,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50
        )

    result = processor.decode(outputs[0], skip_special_tokens=True)

    # Clean response
    if "ASSISTANT:" in result:
        result = result.split("ASSISTANT:")[-1].strip()

    tokens = result.split()

    return result, len(tokens), tokens

In [ ]:
# ── Cell 6: Configuration ─────────────────────────────────────────────────────

import os

prompt = "Describe this image briefly in one short sentence "

In [ ]:
# ── Cell 7: Process all images and save CSV ───────────────────────────────

import csv

output_path = "/content/drive/MyDrive/fire_detection/PickTheModelDataSet(30pics)/phrases_llava.csv"

COLUMNS = [
    "filename",
    "phrase",
    "meaningful_tokens",
    "word_count",
    "meaningful_token_count"
]

with open(output_path, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=COLUMNS)
    writer.writeheader()

    for image_file in image_files:
        image_path = os.path.join(IMAGE_DIR, image_file)

        try:
            result, out_tok_num, tokens = analyze_image(image_path, prompt)

            meaningful_tokens = extract_meaningful_tokens(result)
            words = result.split()

            writer.writerow({
                "filename": image_file,
                "phrase": result,
                "meaningful_tokens": " ".join(meaningful_tokens),
                "word_count": len(words),
                "meaningful_token_count": len(meaningful_tokens)
            })

            print(
                f"{image_file} | {result} | "
                f"{' '.join(meaningful_tokens)} | "
                f"{len(words)} | {len(meaningful_tokens)}"
            )

        except Exception as e:
            print(f"{image_file} | ERROR: {e}")

print(f"\nDone. CSV saved to {output_path}")